downloading the data and looking at the splits

In [1]:
import hashlib
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")
ZIP_PATH = DATA_DIR / "Brain_Tumor_MRI_Dataset_Final.zip"
ROOT = DATA_DIR / "bdneuro"  # ROOT/{split}/{class}/*.jpg
URL = ("https://data.mendeley.com/public-files/datasets/zwr4ntf94j/files/"
       "2d24ea9a-23e3-4a95-898d-06f8cef24c29/file_downloaded")
SHA256 = "0d8fdb6a88b2c65019c11abfd20e0693ffb7b2296955598202303840b9be0c24"

# download once, verify, extract once
DATA_DIR.mkdir(exist_ok=True)
if not ZIP_PATH.exists():
    urllib.request.urlretrieve(URL, ZIP_PATH)
assert hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest() == SHA256, "bad download, delete the zip"
if not (ROOT / "train").exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)

# count images per split and class
counts = pd.DataFrame({
    split.name: {cls.name: sum(1 for _ in cls.iterdir()) for cls in sorted(split.iterdir())}
    for split in ROOT.iterdir() if split.is_dir()
})[["train", "val", "test"]]
counts["total"] = counts.sum(axis=1)
counts.loc["total"] = counts.sum()
counts

,train,val,test,total
glioma,1534,329,328,2191
meningioma,1035,222,221,1478
no_tumor,551,118,118,787
pituitary,1040,223,222,1485
total,4160,892,889,5941


judging balance

In [2]:
# split percentages: how each class (and the whole dataset) is divided over train/val/test
split_pct = counts[["train", "val", "test"]].div(counts["total"], axis=0).mul(100).round(1)
print("Split % per class (rows sum to 100):")
print(split_pct, end="\n\n")

# class balance: what share of each set every class makes up
class_pct = counts.drop(index="total").div(counts.loc["total"], axis=1).mul(100).round(1)
print("Class % within each set (columns sum to 100):")
print(class_pct)

Split % per class (rows sum to 100):
            train   val  test
glioma       70.0  15.0  15.0
meningioma   70.0  15.0  15.0
no_tumor     70.0  15.0  15.0
pituitary    70.0  15.0  14.9
total        70.0  15.0  15.0

Class % within each set (columns sum to 100):
            train   val  test  total
glioma       36.9  36.9  36.9   36.9
meningioma   24.9  24.9  24.9   24.9
no_tumor     13.2  13.2  13.3   13.2
pituitary    25.0  25.0  25.0   25.0


* no_tumor class lacking representation
* decently sized imbalance
* just picking glioma would give ~37% accuracy, accuracy bad metric for this dataset
* per-class recall and confusion matrix will be important
* 